# Elizabeth: weak and strong formulations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/elizabeth-location-models.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup

Colab already provides the general-purpose scientific libraries. This cell ensures the tested Pyomo and HiGHS versions; pip keeps an already-installed matching version. It does not replace Colab's NumPy, pandas or Matplotlib just to match the maintenance environment.


In [ ]:
# Use installed packages; install only missing ones, without version pins.
from importlib.util import find_spec
missing_packages = [name for name in ['pyomo', 'highspy'] if find_spec(name) is None]
if missing_packages:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])


In [ ]:
import pyomo.environ as pyo
from pyomo.opt import assert_optimal_termination
SOLVER = "appsi_highs"
assert pyo.SolverFactory(SOLVER).available(exception_flag=False)


## The same integer decisions, different relaxations
Let $y_j$ indicate whether facility $j$ opens and let $x_{ij}$ assign customer $i$ to facility $j$. Each customer must be assigned once. We compare
$$\sum_i x_{ij}\leq |I|y_j \quad\hbox{and}\quad x_{ij}\leq y_j\;\;\forall i,j.$$
Both are correct for binary variables. The second family implies the first when summed over customers and gives a tighter LP relaxation. This says something about the bound, not a guarantee about solver running time on every instance.


In [ ]:
import numpy as np
rng = np.random.default_rng(42)
facilities = rng.uniform(0,100,(6,2))
customers = rng.uniform(0,100,(24,2))
installation = rng.uniform(100,200,6)
service = np.linalg.norm(customers[:,None,:]-facilities[None,:,:],axis=2)


In [ ]:
def location_model(strong=True, relax=False):
    m = pyo.ConcreteModel('Elizabeth')
    m.I = pyo.RangeSet(0,len(customers)-1)
    m.J = pyo.RangeSet(0,len(facilities)-1)
    domain = pyo.UnitInterval if relax else pyo.Binary
    m.open = pyo.Var(m.J,domain=domain)
    m.assign = pyo.Var(m.I,m.J,domain=domain)
    m.cost = pyo.Objective(expr=pyo.quicksum(float(installation[j])*m.open[j] for j in m.J)
        + pyo.quicksum(float(service[i,j])*m.assign[i,j] for i in m.I for j in m.J))
    @m.Constraint(m.I)
    def once(m,i):
        return pyo.quicksum(m.assign[i,j] for j in m.J) == 1
    if strong:
        @m.Constraint(m.I,m.J)
        def link(m,i,j):
            return m.assign[i,j] <= m.open[j]
    else:
        @m.Constraint(m.J)
        def link(m,j):
            return pyo.quicksum(m.assign[i,j] for i in m.I) <= len(customers)*m.open[j]
    return m

values = {}
for strong in [False,True]:
    for relax in [False,True]:
        model = location_model(strong,relax)
        result = pyo.SolverFactory(SOLVER).solve(model)
        assert_optimal_termination(result)
        values[strong,relax] = pyo.value(model.cost)
print(values)
assert abs(values[False,False]-values[True,False]) < 1e-5
assert values[False,True] <= values[True,True] + 1e-5
assert values[True,True] <= values[True,False] + 1e-5


## Interpret the comparison
Explain why the inequality between the LP bounds has this direction for a minimization model. Change the installation costs and repeat. Do not confuse a stronger formulation with a different underlying facility-location problem.
